# Schema Evolution: Merging Schemas in Parquet and Handling Field Changes

**Module**: 08_data_sources_and_formats  
**Notebook**: 04_schema_evolution.ipynb  
**Environment**: Apache Spark 4.2.1 · Standalone Cluster · PySpark · Python 3.13 · Java 21  

---

## Learning Objectives

- **What will I learn?**: How Spark handles schema evolution over time: adding nullable columns, renaming columns, and using mergeSchema=True in Parquet.
- **Why does this topic exist?**: Production datasets change over time. Ingestion pipelines must accommodate evolving schemas without breaking.
- **What problem does it solve?**: Schema mismatch errors when historical data lacks newly added columns.
- **Where is it used in real-world Data Engineering?**: Long-lived data lakes with evolving business entities.

---

<!-- curriculum-enrichment:v1 -->

## Learning Contract

| Attribute | Value |
|---|---|
| **Priority** | Important |
| **Estimated effort** | 2–3 h |
| **Prerequisites** | Parquet/ORC, partitioned lakes, schema basics |
| **Required depth** | Working knowledge: can add columns and reconcile schemas without breaking readers |
| **Data Engineer** | Evolve schemas safely and understand format-specific limits. |
| **Senior Data Engineer** | Design backward-compatible changes and migration plans. |
| **Spark specialist** | Reason about footer merging, type promotion, and table-format evolution. |
| **Next topic** | `08_data_sources_and_formats/05_reading_and_writing_data.ipynb` |

> **How to use this notebook**: read the problem first, then run the code, then connect the
> mechanism to the real-world cases and production notes. The goal is not to memorize API
> syntax; it is to be able to explain and defend a design decision.

---

## Problem and Motivation

### Why this concept exists

Schemas change: a producer adds a field, a type widens, or a column is renamed. How a format handles that
change determines whether old and new files can be read together. Parquet can merge footers to add nullable
columns, but it does not automatically widen every type, and renaming is not free. Schema evolution is a data
contract problem as much as a format feature.

### What happens if you ignore it

A team can still write code that runs on small data. The failure appears later: a job that
looks correct becomes slow, unstable, or expensive at scale, and the team lacks the vocabulary
to diagnose which Spark mechanism is responsible. This notebook builds that vocabulary and
ties it to observable behavior in the lab.

### A small mental model

Think of the topic as a contract between the logical work you describe and the physical work
Spark performs. The contract has a cost, a failure mode, and a set of trade-offs. The sections
below make each of those explicit.

---

## Core Concepts

### Definitions

| Term | Plain-language meaning | Technical meaning |
|---|---|---|
| **Schema evolution** | Change schema over time | Add, remove, or widen fields while keeping readers working |
| **mergeSchema** | Read union of file schemas | Extra footer reads; can be expensive |
| **Type promotion** | Widen a type | Some promotions are supported; others require rewrite |
| **Nullable column** | New field with nulls | Safest additive change |
| **Column rename** | Change a field name | Usually requires a rewrite or mapping layer |
| **Table format evolution** | Delta/Iceberg features | Often richer than raw Parquet |

### How it works, step by step

1. The writer creates files with a new schema.
2. A reader with `mergeSchema` reads each file footer and unions the schemas.
3. Missing columns are filled with nulls for older files.
4. Type conflicts may fail or require an explicit cast.
5. The merged schema is used for the DataFrame.
6. A table format can track schema metadata centrally and avoid per-file footer merging.

### What to keep in mind

- The mechanism is not free; it consumes CPU, memory, network, or disk.
- The right choice depends on data size, distribution, latency target, and failure tolerance.
- A plan or UI observation is stronger evidence than an intuition about what "should" happen.

## 2. Schema Evolution Rules: Parquet supports adding new nullable columns seamlessly via `mergeSchema`.

### 1. Initialize SparkSession on Standalone Cluster

We connect to `spark://spark-master:7077` with the driver configured to bind to container hostname `jupyter`.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = (
    SparkSession.builder
    .appName("Schema_Evolution_Lab")
    .master("spark://spark-master:7077")
    .config("spark.driver.host", "jupyter")
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark-events")
    .config("spark.sql.warehouse.dir", "file:///opt/spark-data/warehouse")
    .config("spark.hadoop.fs.permissions.umask-mode", "000")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f" SparkSession initialized successfully!")
print(f"Spark Version:      {spark.version}")
print(f"Master URL:         {spark.sparkContext.master}")
print(f"App Name:           {spark.sparkContext.appName}")
print(f"Default Parallelism: {spark.sparkContext.defaultParallelism}")
print(f"Spark UI available at: http://localhost:4040")

In [ ]:
evo_path = "/opt/spark-data/warehouse/schema_evo_test"

# Batch 1 (Version 1 Schema: id, name)
v1 = spark.createDataFrame([(1, "Alice"), (2, "Bob")], ["id", "name"])
v1.write.mode("overwrite").parquet(f"{evo_path}/batch1")

# Batch 2 (Version 2 Schema: id, name, email) - Added new column!
v2 = spark.createDataFrame([(3, "Charlie", "charlie@example.com")], ["id", "name", "email"])
v2.write.mode("overwrite").parquet(f"{evo_path}/batch2")

# Read combined dataset with mergeSchema
combined = spark.read.option("mergeSchema", "true").parquet(f"{evo_path}/*")
combined.printSchema()
combined.show()

---

## Experiment: Reading Without mergeSchema vs With mergeSchema

- **1. What are we testing?**: Observe schema behavior when mergeSchema is false vs true.
- **2. What do we expect?**: Without mergeSchema, Spark infers schema from a single file and may miss new columns.

In [ ]:
print("With mergeSchema=True:")
combined.show()

**3. What happened?**: Missing column in batch 1 was automatically populated with nulls.  
**4. Why did it happen?**: mergeSchema merged StructTypes across Parquet file footers.  
**5. Practical Engineering Lesson**: Enable mergeSchema when querying datasets written under evolving schemas.

---

## Real-World Use Cases

> The scenarios below are illustrative. Company names are omitted, and any numbers are hypothetical unless they are measured in this lab.

### Use Case 1: Adding a nullable field

- **Industry**: SaaS
- **Scenario**: A product team adds an optional `referral_code` field to an event stream.
- **Problem**: Old Parquet files do not contain the field, and readers must not fail.
- **Why this concept is relevant**: Adding a nullable column and reading with schema merging lets old and new files coexist.
- **Implementation idea**: Write new files with the field, read with `mergeSchema`, and verify old rows return null.
- **Expected result**: A backward-compatible change with no backfill.
- **Alternative**: Rewrite the table with the new schema.
- **Trade-offs**: A rewrite is clean but expensive and requires downtime or dual writes.

### Use Case 2: Renaming a column

- **Industry**: Financial services
- **Scenario**: A team renames `amount` to `transaction_amount` for clarity.
- **Problem**: Old files still use the old name, and a naive read returns nulls for the new column.
- **Why this concept is relevant**: A rename is not an additive change; it needs a migration or a view that maps the old name.
- **Implementation idea**: Create a view with the alias during the transition, then rewrite the table.
- **Expected result**: A migration with no broken dashboards.
- **Alternative**: Maintain both columns temporarily.
- **Trade-offs**: Dual columns simplify compatibility but create ambiguity and cleanup work.

---

## Business Value

- Fewer broken dashboards after schema changes.
- Safer producer/consumer contracts.
- Lower migration cost through additive changes.
- Clear ownership of schema compatibility.

> **Note on numbers**: this repository does not claim production benchmarks. If you want to
> quantify a business case, measure the job in your own environment (runtime, shuffle bytes,
> spill, infrastructure cost) before and after the change.

---

## Production Engineering

### What changes when this runs in production

- Prefer additive nullable changes.
- Avoid relying on automatic type widening without testing.
- Treat renames as migrations, not as simple edits.
- Measure the cost of `mergeSchema` on large tables.
- Use a table format if schema evolution is frequent.

### Investigation checklist

- [ ] Open the Spark UI and identify the relevant tab for this topic.
- [ ] Compare the metric before and after a configuration or code change.
- [ ] Check driver and executor logs for warnings or retries.
- [ ] Confirm that the fix is measured, not assumed.

### Common production mistakes

1. Copying a local-lab setting into a production cluster without sizing it.
2. Optimizing a single job without checking the downstream SLA and data-quality impact.
3. Treating a successful run as proof of correctness; verify row counts, keys, and invariants.
4. Forgetting that retries and backfills must be idempotent.

### Experiment: Read old and new files together

**Steps**

1. Write a Parquet dataset with two columns.
2. Append a file with an additional nullable column.
3. Read with and without `mergeSchema` and compare the resulting schema and null counts.

**What to observe**: Without merging, the reader uses one file's schema; with merging, the new column appears and old rows are null.

**Healthy result**: The merged read returns the expected schema and values.

**Problematic result**: A type conflict causes an error or silent nulls.

**Engineering takeaway**: Test evolution against real old files, not only new ones.

---

## Trade-offs and Alternatives

### Advantages

- Additive changes are cheap
- Old files remain readable
- Table formats add metadata

### Limitations

- Footer merging can be expensive
- Renames are not automatic
- Type conflicts can break reads

### When to use it

- Evolving event streams
- Multi-source lakes
- Long-lived tables

### When not to use it

- Silent type changes
- Frequent renames
- Assuming all formats behave the same

### Alternatives inside Spark

- mergeSchema
- Parquet/ORC readers
- Table formats

### Alternatives outside Spark

- Schema registries
- Avro evolution
- Contract testing

### Performance implications

Every choice has a cost. Measure runtime, shuffle read/write, spill, GC time, and task
duration percentiles before and after the change. A local four-core cluster is useful for
observing mechanics, but it is not a production benchmark.

---

## Practical Exercises

### Exercise 1 (Beginner): Check Nulls in New Column
**Problem Statement**: Filter rows where email is null in evolved schema.  
**Input / Context**: combined

In [ ]:
# Write your solution for Exercise 1 here

### Exercise 2 (Intermediate): Global mergeSchema Config
**Problem Statement**: Print spark.sql.parquet.mergeSchema config.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 2 here

### Exercise 3 (Advanced): Add Nested Field
**Problem Statement**: Simulate adding a nested struct field.  
**Input / Context**: Provided dataset

In [ ]:
# Write your solution for Exercise 3 here

---

## Production Debugging Exercise

### The new column is always null

**Scenario**: A newly added column returns null for all rows after deployment.

**Your task**: identify the most likely root cause, propose one diagnostic step, and propose
one fix. Write your answer before opening the model approach.

**Hints**

1. Check whether the reader uses a fixed schema.
2. Verify that `mergeSchema` is enabled for the read.
3. Confirm the new files actually contain the column and the write was not filtered.

<details>
<summary>Model approach (open after trying)</summary>

1. Reproduce the issue with the smallest possible dataset.
2. Use the Spark UI and the physical plan to locate the stage or operator that dominates.
3. Change one variable at a time and re-measure.
4. Record the before/after metric so the fix is defensible.

The exact fix depends on the evidence. The important habit is to separate allocation,
scheduling, shuffle, memory, and data-quality causes instead of guessing.

</details>

---

## Reference Solutions

<details>
<summary><b>Click to expand full solutions</b></summary>

#### Solution 1 (Check Nulls in New Column):
```python
combined.filter('email IS NULL').show()
```

#### Solution 2 (Global mergeSchema Config):
```python
print(spark.conf.get('spark.sql.parquet.mergeSchema','false'))
```

#### Solution 3 (Add Nested Field):
```python
print('Nested structs can be merged with Delta Lake / Iceberg')
```

</details>

---

## Module Summary & Interview Guide

### Key Takeaways
- **mergeSchema**: Merges disparate Parquet schemas across files.
- **Backwards Compatibility**: New columns must be nullable.

### Essential Technical Terms
- `Schema Evolution`: Adapting to changing data definitions over time.
- `mergeSchema`: Parquet read option combining file footers.

### Common Engineering Mistakes
- **Changing column data types**: Incompatible type changes (e.g. String to Int) break mergeSchema.

### Interview Questions on This Topic
- **Q: How does Spark handle missing columns in older Parquet files when mergeSchema is enabled?**
  - *A*: It populates the missing column with null values for all historical rows.


### More Interview Questions

- **Q: How does Parquet handle a new nullable column?**
  - *A*: With schema merging enabled, the reader unions file schemas and fills the missing column with nulls for older files.
- **Q: Why is renaming a column harder than adding one?**
  - *A*: Adding a nullable field is backward-compatible; renaming changes the field identity, so old files still expose the old name and require a mapping or rewrite.

### What to Study Next
Open 05_reading_and_writing_data.ipynb

---

### Tear Down SparkSession
Always close your session when finishing to release executor threads and driver ports.

In [ ]:
spark.stop()
print('SparkSession successfully stopped.')